# RTMDet-Pose - Custom Training on Google Colab

Single-stage, anchor-free, real-time multi-person pose estimation.  
**Same dataset format and workflow as DETRPose / RTMOPose** — no DCC, LTRB bbox encoding.

| Model | Backbone | Params | ~Speed (A100) |
|-------|----------|--------|---------------|
| N     | HGNetV2-B0 nano  | ~4 M  | ~25 s/epoch |
| S     | HGNetV2-B0 small | ~8 M  | ~40 s/epoch |
| M     | HGNetV2-B2       | ~18 M | ~65 s/epoch |
| L     | HGNetV2-B4       | ~45 M | ~110 s/epoch |

## Workflow
1. **Settings** – fill in your Google Drive paths, model size, training options  
2. **Mount Drive** – authenticate Google Drive  
3. **Check GPU** – verify A100/T4 runtime  
4. **Initialize Environment** – install the wheel and dependencies (cached for repeat runs)  
5. **Fetch Dataset** – extract COCO-format data from Drive  
6. **Fetch Weights** – (optional) copy pretrained backbone weights  
7. **Train** – launch training with live output  
8. **Save to Drive** – copy results to Google Drive

## Settings
Fill in the paths and options below before running any other cells.

In [ ]:

# ── Google Drive paths ────────────────────────────────────────────────────────
PROJECT_NAME = 'Dev Project/keypoints'   # Folder inside MyDrive/Datasets/
OBJECT_NAME  = 'Test_Coco/coco_v2'          # Dataset folder name (contains train.7z, val.7z)

# ── Model ─────────────────────────────────────────────────────────────────────
# 'n' = nano  |  's' = small  |  'm' = medium  |  'l' = large  |  'x' = extra-large
MODEL_SIZE = 's'
MODEL_TASK = 'pose'  # 'pose' or 'detect'

# ── Input image size ──────────────────────────────────────────────────────────
# All train/val/test pipelines are forced to this size from the notebook.
# If source images already match this size, resize work is skipped by transform fast-path.
IMAGE_SIZE = 640

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS         = 200    # Override epoch count (set to None to use config default)
BATCH_SIZE     = 32     # Override total batch size (set to None to use config default)
NUM_WORKERS    = 8      # DataLoader worker processes per split
EVAL_INTERVAL  = 5      # Run COCO evaluation every N epochs (1 = every epoch)
USE_AMP        = True   # Mixed-precision training (fp16) — ~1.5-2x faster
BACKUP_EVERY_N_EPOCHS = 10   # Back up output to Google Drive every N epochs

# ── Resume training ───────────────────────────────────────────────────────────
RESUME_TRAINING   = False
RESUME_CHECKPOINT = 'checkpoint.pth'   # filename inside output dir on Google Drive

# ── Pretrained backbone weights (optional) ────────────────────────────────────
# Set to None to auto-download, or a GDrive filename inside weights/ folder
PRETRAIN_DFINE_WEIGHTS = None

# ── Wheel source ──────────────────────────────────────────────────────────────
GDRIVE_WHEEL_PATH = '/content/drive/MyDrive/Installer/Dependencies/PhantomFactory/Application/visionhub-1.0.0-py3-none-any.whl'

# ── Cached packages (speeds up environment setup on repeat runs) ──────────────
CACHED_RTDETPOSE_VERSION = '1.0.0'

# ─────────────────────────────────────────────────────────────────────────────
# Derived paths (do not edit below)
# ─────────────────────────────────────────────────────────────────────────────
import os

REPO_NAME          = 'visionhub_runtime'
REPO_LOCAL_PATH    = f'/content/{REPO_NAME}'
DATA_LOCAL_PATH    = '/content/coco_data'
OUTPUT_LOCAL_PATH  = os.path.join(REPO_LOCAL_PATH, 'output')

GDRIVE_DATASET_PATH   = f'/content/drive/MyDrive/Datasets/{PROJECT_NAME}/{OBJECT_NAME}'
GDRIVE_WEIGHTS_PATH   = os.path.join(GDRIVE_DATASET_PATH, 'weight')
GDRIVE_INSTALLER_PATH = '/content/drive/MyDrive/Installer/Colab'
GDRIVE_OUTPUT_PATH    = os.path.join(GDRIVE_DATASET_PATH, 'training_runs')
MODEL_FAMILY = 'rtmdetpose' if MODEL_TASK == 'pose' else 'rtmdetdet'
GDRIVE_BACKUP_PATH    = os.path.join(GDRIVE_DATASET_PATH, f'training_runs_backup_{MODEL_FAMILY}')

VENV_PATH        = '/content/venv'
LOCAL_CACHE_PATH = '/content/cache'
CONFIG_REL_PATH = f'configs/{MODEL_FAMILY}/{MODEL_FAMILY}_hgnetv2_{MODEL_SIZE}_custom.py'
CONFIG_FILE = os.path.join(VENV_PATH, 'lib', 'python3.10', 'site-packages', CONFIG_REL_PATH)
OUTPUT_DIR  = f'output/{MODEL_FAMILY}_hgnetv2_{MODEL_SIZE}_custom'

CACHED_SITE_PACKAGES_ARCHIVE = f'site-packages_detrpose-{CACHED_RTDETPOSE_VERSION}.7z'
CACHED_SITE_PACKAGES_PATH    = os.path.join(GDRIVE_INSTALLER_PATH, CACHED_SITE_PACKAGES_ARCHIVE)

CUDA_VERSION      = '12.4.0'
CUDNN_VERSION     = '9.16.0'
CACHED_CUDA_ARCHIVE = f'cuda-{CUDA_VERSION}-cudnn-{CUDNN_VERSION}.7z'
CACHED_CUDA_PATH    = os.path.join(GDRIVE_INSTALLER_PATH, CACHED_CUDA_ARCHIVE)

PRETRAIN_LOCAL_PATH = None

print(f'Model task    : {MODEL_TASK}')
print(f'Model family  : {MODEL_FAMILY}')
print(f'Model size    : {MODEL_SIZE}')
print(f'Image size    : {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Config file   : {CONFIG_FILE}')
print(f'Output dir    : {OUTPUT_DIR}')
print(f'Dataset       : {GDRIVE_DATASET_PATH}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'Eval interval : every {EVAL_INTERVAL} epoch(s)')


## Mount Google Drive

In [ ]:
from google.colab import drive  # type: ignore
drive.mount('/content/drive')

## Check Colab Runtime Type

In [ ]:
import subprocess
print('=== Checking GPU ===')
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
        capture_output=True, text=True, check=True
    )
    print(result.stdout.strip())
except Exception as e:
    print(f'nvidia-smi failed: {e}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

## Initialize Environment
Installs the `visionhub` wheel from Google Drive, then installs all dependencies.  
On repeat runs a cached site-packages archive on Google Drive is used to skip pip install (~30 s vs ~5 min).

In [ ]:

import os
import shutil
import subprocess
from pathlib import Path

os.makedirs(LOCAL_CACHE_PATH, exist_ok=True)

pip_path    = os.path.join(VENV_PATH, 'bin', 'pip')
python_path = os.path.join(VENV_PATH, 'bin', 'python')

# ── 1. Prepare runtime workspace and wheel source ────────────────────────────
print('=== Preparing wheel-based runtime workspace ===')
os.makedirs(REPO_LOCAL_PATH, exist_ok=True)
if not os.path.isfile(GDRIVE_WHEEL_PATH):
    raise FileNotFoundError(f'Wheel file not found on Google Drive: {GDRIVE_WHEEL_PATH}')
wheel_uri = Path(GDRIVE_WHEEL_PATH).resolve().as_uri()
wheel_spec = f'visionhub[export] @ {wheel_uri}'
wheel_base_spec = f'visionhub @ {wheel_uri}'
print(f'✓ Runtime workspace ready at {REPO_LOCAL_PATH}')
print(f'✓ Wheel source: {GDRIVE_WHEEL_PATH}')


In [ ]:

import os
import shutil
import subprocess

# ── 2. Create venv with Python 3.10 ─────────────────────────────────────────
print('\n=== Installing Python 3.10 ===')
subprocess.run(['apt-get', 'update'], check=True, capture_output=True)
subprocess.run(['apt-get', 'install', '-y', 'python3.10', 'python3.10-venv'],
               check=True, capture_output=True)
print('✓ Python 3.10 ready')

print(f'\n=== Creating virtual environment in {VENV_PATH} ===')
subprocess.run(['python3.10', '-m', 'venv', VENV_PATH], check=True, capture_output=True)
print('✓ Virtual environment created')

# ── 3. CUDA / cuDNN (cached or fresh) ────────────────────────────────────────
use_cached_cuda = os.path.exists(CACHED_CUDA_PATH)

if use_cached_cuda:
    print(f'\n=== Using cached CUDA/cuDNN from Google Drive ===')
    local_cuda_archive = os.path.join(LOCAL_CACHE_PATH, CACHED_CUDA_ARCHIVE)
    shutil.copy(CACHED_CUDA_PATH, local_cuda_archive)
    subprocess.run(['7z', 'x', local_cuda_archive, f'-o{VENV_PATH}', '-y'],
                   check=True, capture_output=True)
    os.remove(local_cuda_archive)
    print('✓ Cached CUDA/cuDNN extracted')
else:
    print('\n=== Installing CUDA 12.4 and cuDNN 9.16 from scratch ===')
    subprocess.run(['wget', '-q',
        'https://developer.download.nvidia.com/compute/cuda/12.4.0/local_installers/cuda_12.4.0_550.54.14_linux.run'],
        check=True, capture_output=True)
    subprocess.run(['chmod', '+x', 'cuda_12.4.0_550.54.14_linux.run'], check=True, capture_output=True)
    os.makedirs('/content/venv/cuda', exist_ok=True)
    subprocess.run(['./cuda_12.4.0_550.54.14_linux.run', '--silent', '--toolkit',
                    '--installpath=/content/venv/cuda'], check=True, capture_output=True)

    subprocess.run(['wget', '-q',
        'https://developer.download.nvidia.com/compute/cudnn/redist/cudnn/linux-x86_64/cudnn-linux-x86_64-9.16.0.29_cuda12-archive.tar.xz'],
        check=True, capture_output=True)
    subprocess.run(['tar', '-xf', 'cudnn-linux-x86_64-9.16.0.29_cuda12-archive.tar.xz'],
                   check=True, capture_output=True)
    subprocess.run('cp -r cudnn-linux-x86_64-9.16.0.29_cuda12-archive/include/* /content/venv/cuda/include/',
                   shell=True, check=True, capture_output=True)
    subprocess.run('cp -r cudnn-linux-x86_64-9.16.0.29_cuda12-archive/lib/* /content/venv/cuda/lib64/',
                   shell=True, check=True, capture_output=True)
    for f in ['cuda_12.4.0_550.54.14_linux.run',
              'cudnn-linux-x86_64-9.16.0.29_cuda12-archive.tar.xz',
              'cudnn-linux-x86_64-9.16.0.29_cuda12-archive']:
        subprocess.run(['rm', '-rf', f], capture_output=True)
    print('✓ CUDA and cuDNN installed')

# ── 4. Python packages (cached or fresh) ────────────────────────────────────
use_cached_packages = os.path.exists(CACHED_SITE_PACKAGES_PATH)

if use_cached_packages:
    print(f'\n=== Using cached site-packages from Google Drive ===')
    local_pkg_archive = os.path.join(LOCAL_CACHE_PATH, CACHED_SITE_PACKAGES_ARCHIVE)
    shutil.copy(CACHED_SITE_PACKAGES_PATH, local_pkg_archive)
    extract_target = os.path.join(VENV_PATH, 'lib', 'python3.10')
    subprocess.run(['7z', 'x', local_pkg_archive, f'-o{extract_target}', '-aoa'],
                   check=True, capture_output=True)
    os.remove(local_pkg_archive)
    print('✓ Cached site-packages extracted')
else:
    print('\n=== Installing packages from scratch ===')
    subprocess.run([pip_path, 'install', '--upgrade', 'pip'],
                   check=True, capture_output=True)
    # PyTorch (CUDA 12.4)
    subprocess.run(
        [pip_path, 'install', 'torch==2.5.1', 'torchvision==0.20.1', 'torchaudio==2.5.1',
         '--index-url', 'https://download.pytorch.org/whl/cu124'],
        check=True, capture_output=True
    )
    # visionhub wheel and Python dependencies
    subprocess.run(
        [pip_path, 'install', wheel_spec],
        check=True, capture_output=True
    )
    print('✓ Packages installed')

print('\n=== Ensuring wheel install is active ===')
subprocess.run([pip_path, 'install', '--no-deps', '--force-reinstall', wheel_base_spec],
               check=True, capture_output=True)
print('✓ visionhub wheel installed')

# Always remove wandb to prevent experiment tracking overhead
subprocess.run([pip_path, 'uninstall', '-y', 'wandb'], capture_output=True)

if os.path.exists(LOCAL_CACHE_PATH):
    shutil.rmtree(LOCAL_CACHE_PATH)

# ── 5. Verify ────────────────────────────────────────────────────────────────
print('\n=== Verifying installation ===')
verify_env = os.environ.copy()
verify_env['MPLBACKEND'] = 'Agg'

result = subprocess.run(
    [python_path, '-c', 'import torch, visionhub; print(f"PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}, visionhub {visionhub.__version__}")'],
    capture_output=True, text=True, env=verify_env
)
print(result.stdout.strip())

cuda_bin = '/content/venv/cuda/bin/nvcc'
if os.path.exists(cuda_bin):
    r = subprocess.run([cuda_bin, '--version'], capture_output=True, text=True)
    lines = [l for l in r.stdout.split('\n') if 'release' in l]
    if lines:
        print(f'CUDA: {lines[0].split("release")[-1].strip().rstrip(",")}')

print('✓ Environment ready')


## Fetch Dataset from Google Drive
Copies `train.7z` and `val.7z` from Google Drive and extracts them into the expected COCO layout.

In [ ]:
import os
import shutil
import subprocess

# Clean up any previous local data
if os.path.exists(DATA_LOCAL_PATH):
    shutil.rmtree(DATA_LOCAL_PATH)
os.makedirs(DATA_LOCAL_PATH, exist_ok=True)

def copy_and_extract(split: str):
    archive_name = f'{split}.7z'
    gdrive_archive = os.path.join(GDRIVE_DATASET_PATH, archive_name)
    local_archive  = os.path.join(DATA_LOCAL_PATH, archive_name)
    dest_dir       = os.path.join(DATA_LOCAL_PATH, split)

    if not os.path.isfile(gdrive_archive):
        raise FileNotFoundError(
            f"Expected archive not found on Google Drive:\n  {gdrive_archive}\n"
            f"Please upload {archive_name} to: {GDRIVE_DATASET_PATH}"
        )

    print(f'Copying {archive_name} from Google Drive ...')
    shutil.copy(gdrive_archive, local_archive)
    print(f'✓ Copied ({os.path.getsize(local_archive) / 1024**2:.1f} MB)')

    print(f'Extracting {archive_name} ...')
    os.makedirs(dest_dir, exist_ok=True)
    result = subprocess.run(
        ['7z', 'x', local_archive, f'-o{dest_dir}', '-y'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f'Extraction failed:\n{result.stderr}')

    os.remove(local_archive)

    # Verify expected files
    ann_file  = os.path.join(dest_dir, 'coco_instances.json')
    img_dir   = os.path.join(dest_dir, 'images')
    n_images  = len(os.listdir(img_dir)) if os.path.isdir(img_dir) else 0
    ann_ok    = os.path.isfile(ann_file)
    print(f'✓ {split}: {n_images} images, annotation file {"found" if ann_ok else "MISSING"}')

    if not ann_ok:
        raise FileNotFoundError(
            f'coco_instances.json not found in {dest_dir}.\n'
            'Make sure the archive extracts to: {split}/coco_instances.json'
        )

print('=== Fetching training data ===')
copy_and_extract('train')

print('\n=== Fetching validation data ===')
copy_and_extract('val')

# Read and print dataset info
import json
with open(os.path.join(DATA_LOCAL_PATH, 'train', 'coco_instances.json')) as f:
    ann = json.load(f)

cats = ann['categories']
cat_ids = sorted(c['id'] for c in cats)
num_classes     = len(cat_ids)
num_body_points = max((len(c.get('keypoints', [])) for c in cats), default=0)

print(f'\n=== Dataset summary ===')
print(f'  Categories       : {len(cats)}')
print(f'  Category IDs     : {cat_ids}')
print(f'  Contiguous classes: {num_classes}')
print(f'  Keypoints/instance: {num_body_points}')
print(f'  Training images  : {len(ann["images"])}')
print(f'  Training annots  : {len(ann["annotations"])}')

## Fetch Pretrained Weights (Optional)
Copies backbone/HGNetV2 pretrain weights from Google Drive.  
Upload weights to `MyDrive/Datasets/{PROJECT_NAME}/{OBJECT_NAME}/weight/` and set `PRETRAIN_DFINE_WEIGHTS` in Settings.

In [ ]:
import os
import shutil

PRETRAIN_LOCAL_PATH = None

if PRETRAIN_DFINE_WEIGHTS:
    gdrive_weight = os.path.join(GDRIVE_WEIGHTS_PATH, PRETRAIN_DFINE_WEIGHTS)
    if not os.path.isfile(gdrive_weight):
        raise FileNotFoundError(f'Pretrain weights not found: {gdrive_weight}')
    local_weights_dir = os.path.join(REPO_LOCAL_PATH, 'weights', 'hgnetv2')
    os.makedirs(local_weights_dir, exist_ok=True)
    dst = os.path.join(local_weights_dir, PRETRAIN_DFINE_WEIGHTS)
    shutil.copy2(gdrive_weight, dst)
    PRETRAIN_LOCAL_PATH = dst
    print(f'Pretrain weights copied to: {dst}')
else:
    print('No pretrain weights specified — backbone will auto-download if needed.')

## (Optional) Reinstall Wheel
Run this cell to force-reinstall the wheel from Google Drive without restarting the runtime.

In [ ]:
import os
import subprocess
from pathlib import Path

pip_path = os.path.join(VENV_PATH, 'bin', 'pip')
if not os.path.isfile(GDRIVE_WHEEL_PATH):
    raise FileNotFoundError(f'Wheel file not found: {GDRIVE_WHEEL_PATH}')
wheel_uri = Path(GDRIVE_WHEEL_PATH).resolve().as_uri()
wheel_base_spec = f'visionhub @ {wheel_uri}'
subprocess.run([pip_path, 'install', '--no-deps', '--force-reinstall', wheel_base_spec], check=True)
print(f'Reinstalled wheel from {GDRIVE_WHEEL_PATH}')

## Train Model
Runs `visionhub-train` with the selected RTMDet-Pose config.  
Output is streamed live and periodically backed up to Google Drive.

In [ ]:
import datetime
import os
import shutil
import subprocess
from zoneinfo import ZoneInfo

python_path = os.path.join(VENV_PATH, 'bin', 'python')

# ── Build training command ────────────────────────────────────────────────────
train_cli = os.path.join(VENV_PATH, 'bin', 'visionhub-train')
cmd = [train_cli, '--config_file', CONFIG_FILE, '--data_root', DATA_LOCAL_PATH]

options = [f'training_params.output_dir="{OUTPUT_DIR}"']
options.append(f'training_params.backup_every_n_epochs={BACKUP_EVERY_N_EPOCHS}')
options.append(f'training_params.gdrive_backup_path="{GDRIVE_BACKUP_PATH}"')

# Force a single image size from notebook settings for all splits.
# The CLI runtime override updates the current transform layout safely.

if EPOCHS is not None:
    options.append(f'training_params.epochs={EPOCHS}')
    lr_milestone = int(EPOCHS * 0.8)
    options.append(f'lr_scheduler.milestones=[{lr_milestone}]')

if BATCH_SIZE is not None:
    options.append(f'dataset_train.total_batch_size={BATCH_SIZE}')
    options.append(f'dataset_val.total_batch_size={BATCH_SIZE}')
    options.append(f'dataset_test.total_batch_size={BATCH_SIZE}')

if NUM_WORKERS is not None:
    options.append(f'dataset_train.num_workers={NUM_WORKERS}')
    options.append(f'dataset_val.num_workers={NUM_WORKERS}')
    options.append(f'dataset_test.num_workers={NUM_WORKERS}')

if EVAL_INTERVAL is not None:
    options.append(f'training_params.eval_interval={EVAL_INTERVAL}')

cmd += ['--options'] + options
cmd += ['--image_size', str(IMAGE_SIZE)]

if USE_AMP:
    cmd.append('--amp')

if RESUME_TRAINING:
    resume_ckpt = os.path.join(REPO_LOCAL_PATH, OUTPUT_DIR, RESUME_CHECKPOINT)
    if not os.path.isfile(resume_ckpt):
        gdrive_ckpt = os.path.join(GDRIVE_BACKUP_PATH, RESUME_CHECKPOINT)
        if os.path.isfile(gdrive_ckpt):
            os.makedirs(os.path.dirname(resume_ckpt), exist_ok=True)
            shutil.copy(gdrive_ckpt, resume_ckpt)
            print('Checkpoint copied from Google Drive backup')
        else:
            raise FileNotFoundError(
                f'Resume checkpoint not found:\n'
                f'  Local  : {resume_ckpt}\n'
                f'  GDrive : {gdrive_ckpt}'
            )
    cmd += ['--resume', resume_ckpt]
    print(f'Resuming from: {resume_ckpt}')

if PRETRAIN_LOCAL_PATH:
    cmd += ['--pretrain', PRETRAIN_LOCAL_PATH]

# ── Environment ───────────────────────────────────────────────────────────────
train_env = os.environ.copy()
train_env['CUDA_HOME']          = '/content/venv/cuda'
train_env['PATH']               = f"/content/venv/cuda/bin:{train_env.get('PATH', '')}"
train_env['LD_LIBRARY_PATH']    = f"/content/venv/cuda/lib64:{train_env.get('LD_LIBRARY_PATH', '')}"
train_env['CUDNN_PATH']         = '/content/venv/cuda'
train_env['RTMDETPOSE_DATA_ROOT'] = DATA_LOCAL_PATH
train_env['RTMDETDET_DATA_ROOT']  = DATA_LOCAL_PATH
train_env['RTMDET_DATA_ROOT']     = DATA_LOCAL_PATH
train_env['MPLBACKEND']         = 'Agg'

# ── Run training ──────────────────────────────────────────────────────────────
tz = ZoneInfo('Africa/Johannesburg')
start_dt = datetime.datetime.now(datetime.timezone.utc).astimezone(tz)
print(f'Start time (+02:00): {start_dt.strftime("%Y/%m/%d %H:%M:%S")}')
print(f'Dataset root      : {DATA_LOCAL_PATH}')
print(f'Train ann exists  : {os.path.isfile(os.path.join(DATA_LOCAL_PATH, "train", "coco_instances.json"))}')
print(f'Val ann exists    : {os.path.isfile(os.path.join(DATA_LOCAL_PATH, "val", "coco_instances.json"))}')
print(f'RTMDETDET_DATA_ROOT: {train_env.get("RTMDETDET_DATA_ROOT")}')
print(f'RTMDET_DATA_ROOT   : {train_env.get("RTMDET_DATA_ROOT")}')
print(f'Command: {" ".join(cmd)}')
print('=' * 80)

process = subprocess.Popen(
    cmd,
    cwd=REPO_LOCAL_PATH,
    env=train_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
returncode = process.returncode

end_dt = datetime.datetime.now(datetime.timezone.utc).astimezone(tz)
print('\n' + '=' * 80)
print(f'End time (+02:00): {end_dt.strftime("%Y/%m/%d %H:%M:%S")}')
print(f'Elapsed: {end_dt - start_dt}')
if returncode == 0:
    print('Training completed successfully!')
else:
    print(f'Training exited with code {returncode}')

## (Dev Only) Archive Packages for Cached Retrieval
Run this once after a successful fresh install to create a cached archive on Google Drive.  
Subsequent runs will restore from this archive instead of pip-installing (~30 s vs ~5 min).

In [ ]:
import os
import subprocess

os.makedirs(GDRIVE_INSTALLER_PATH, exist_ok=True)

def _archive_to_gdrive(src_path, gdrive_archive_path):
    print(f'Archiving {src_path} -> {gdrive_archive_path}')
    subprocess.run(
        ['7z', 'a', '-t7z', '-mx=3', gdrive_archive_path, '.'],
        cwd=src_path,
        check=True
    )
    print(f'Archived to {gdrive_archive_path}')

site_packages_path = os.path.join(VENV_PATH, 'lib', 'python3.10', 'site-packages')
_archive_to_gdrive(site_packages_path, CACHED_SITE_PACKAGES_PATH)